# Final Notebook - Support Vector Machine (SVM)
**Penanggung Jawab:** Kevin Wirya Valerian (13524019)  
**Kelompok:** G03 - FiveAgents  
**Algoritma:** Support Vector Machine (Linear, Polynomial, Gaussian/RBF) dari Nol + Komparasi scikit-learn

## 1. inisialisasi lingkungan dan pembacaan data

In [ ]:
from pathlib import Path
import sys
import time
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.svm import SVC

# deteksi lokasi root repositori secara fleksibel
curr = Path('.').resolve()
REPO_ROOT = None
for cand in [curr, curr / 'IF3170_G03_FiveAgents', curr.parent, curr.parent / 'IF3170_G03_FiveAgents', curr.parents[1], curr.parents[2]]:
    if (cand / 'data' / 'train.csv').exists() and (cand / 'src').exists():
        REPO_ROOT = cand
        break

if REPO_ROOT is None:
    raise FileNotFoundError('root repositori tidak ditemukan')

sys.path.insert(0, str(REPO_ROOT / 'src' / 'experiments' / 'svm'))
from preprocessing import SVMPreprocessor, make_contract_split
from svm_scratch import SVMFromScratch

print(f'repo root: {REPO_ROOT}')
train_df = pd.read_csv(REPO_ROOT / 'data' / 'train.csv', dtype={'id': str})
test_df = pd.read_csv(REPO_ROOT / 'data' / 'test.csv', dtype={'id': str})

print(f'train shape: {train_df.shape}, test shape: {test_df.shape}')

repo root: D:\IF_Semester 5\AI\Tubes 2\IF3170_G03_FiveAgents
train shape: (27373, 19), test shape: (6844, 18)


## 2. split validasi stratified dan prapemrosesan

In [ ]:
train_idx, valid_idx = make_contract_split(train_df, seed=42)
df_train = train_df.iloc[train_idx].reset_index(drop=True)
df_valid = train_df.iloc[valid_idx].reset_index(drop=True)

y_train = df_train["is_canceled"].to_numpy(dtype=np.int64)
y_valid = df_valid["is_canceled"].to_numpy(dtype=np.int64)

print(f"train_idx: {len(train_idx)} baris, distribusi kelas: {np.bincount(y_train)}")
print(f"valid_idx: {len(valid_idx)} baris, distribusi kelas: {np.bincount(y_valid)}")

# prapemrosesan fit hanya pada df_train
prep = SVMPreprocessor(seed=42)
prep.fit(df_train)

X_train = prep.transform(df_train)
X_valid = prep.transform(df_valid)
X_test = prep.transform(test_df)

print(f"matriks fitur: X_train={X_train.shape}, X_valid={X_valid.shape}, X_test={X_test.shape}")
print(f"jumlah fitur akhir: {len(prep.feature_names_)}")

train_idx: 21898 baris, distribusi kelas: [14723  7175]
valid_idx: 5475 baris, distribusi kelas: [3681 1794]
matriks fitur: X_train=(21898, 90), X_valid=(5475, 90), X_test=(6844, 90)
jumlah fitur akhir: 90


## 3. evaluasi 3 kernel svm dari nol pada data validasi

In [ ]:
def evaluate_model(name, y_true, y_pred, fit_time=0.0, pred_time=0.0):
    macro_f1 = f1_score(y_true, y_pred, labels=[0, 1], average="macro", zero_division=0)
    prec, rec, f1_cls, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f"=== {name} ===")
    print(f"macro f1 : {macro_f1:.4f}")
    print(f"kelas 0  : prec={prec[0]:.4f}, rec={rec[0]:.4f}, f1={f1_cls[0]:.4f}")
    print(f"kelas 1  : prec={prec[1]:.4f}, rec={rec[1]:.4f}, f1={f1_cls[1]:.4f}")
    print(f"confusion matrix:\n{cm}")
    if fit_time > 0:
        print(f"waktu fit: {fit_time:.2f}s | waktu predict: {pred_time:.3f}s")
    print()
    return {
        "name": name,
        "macro_f1": macro_f1,
        "f1_0": f1_cls[0],
        "f1_1": f1_cls[1],
        "fit_time": fit_time,
        "pred_time": pred_time,
        "cm": cm
    }

models_scratch = {
    "Linear (from-scratch)": SVMFromScratch(C=0.5, kernel="linear", threshold=-0.03, random_state=42),
    "Polynomial (from-scratch)": SVMFromScratch(C=2.0, kernel="poly", degree=2, coef0=1.0, class_weight="balanced", threshold=0.29, random_state=42),
    "Gaussian RBF (from-scratch)": SVMFromScratch(C=1.0, kernel="rbf", gamma="scale", threshold=-0.16, random_state=42),
}

results_scratch = {}
for name, m in models_scratch.items():
    t0 = time.perf_counter()
    m.fit(X_train, y_train)
    fit_t = time.perf_counter() - t0
    
    t1 = time.perf_counter()
    preds = m.predict(X_valid)
    pred_t = time.perf_counter() - t1
    
    res = evaluate_model(name, y_valid, preds, fit_t, pred_t)
    res["slack_valid"] = m.calculate_slack(X_valid, y_valid)
    res["support_vectors"] = m.n_support_
    results_scratch[name] = res

=== Linear (from-scratch) ===
macro f1 : 0.8574
kelas 0  : prec=0.9017, rec=0.9142, f1=0.9079
kelas 1  : prec=0.8187, rec=0.7954, f1=0.8069
confusion matrix:
[[3365  316]
 [ 367 1427]]
waktu fit: 1.84s | waktu predict: 0.002s

=== Polynomial (from-scratch) ===
macro f1 : 0.8520
kelas 0  : prec=0.8860, rec=0.9313, f1=0.9081
kelas 1  : prec=0.8425, rec=0.7542, f1=0.7959
confusion matrix:
[[3428  253]
 [ 441 1353]]
waktu fit: 9.62s | waktu predict: 0.213s

=== Gaussian RBF (from-scratch) ===
macro f1 : 0.8546
kelas 0  : prec=0.9042, rec=0.9055, f1=0.9048
kelas 1  : prec=0.8055, rec=0.8032, f1=0.8044
confusion matrix:
[[3333  348]
 [ 353 1441]]
waktu fit: 8.03s | waktu predict: 0.359s



## 4. komparasi langsung dengan scikit-learn (SVC)

In [ ]:
sklearn_models = {
    "Linear (scikit-learn)": SVC(C=0.5, kernel="linear", random_state=42),
    "Polynomial (scikit-learn)": SVC(C=2.0, kernel="poly", degree=2, coef0=1.0, class_weight="balanced", random_state=42),
    "Gaussian RBF (scikit-learn)": SVC(C=1.0, kernel="rbf", gamma="scale", random_state=42),
}

results_sklearn = {}
for name, m in sklearn_models.items():
    t0 = time.perf_counter()
    m.fit(X_train, y_train)
    fit_t = time.perf_counter() - t0
    
    t1 = time.perf_counter()
    scores = m.decision_function(X_valid)
    pred_t = time.perf_counter() - t1
    
    best_f1, best_th = -1, 0
    for th in np.linspace(-1.0, 1.0, 101):
        f = f1_score(y_valid, (scores >= th).astype(int), average="macro", zero_division=0)
        if f > best_f1:
            best_f1, best_th = f, th
    preds = (scores >= best_th).astype(int)
    
    res = evaluate_model(name, y_valid, preds, fit_t, pred_t)
    results_sklearn[name] = res

# tabel perbandingan head-to-head
summary_rows = []
for k in ["Linear", "Polynomial", "Gaussian RBF"]:
    sc_res = results_scratch[f"{k} (from-scratch)"]
    sk_res = results_sklearn[f"{k} (scikit-learn)"]
    summary_rows.append({
        "Kernel": k,
        "Macro F1 (Scratch)": round(sc_res["macro_f1"], 4),
        "Macro F1 (Sklearn)": round(sk_res["macro_f1"], 4),
        "Fit Time Scratch (s)": round(sc_res["fit_time"], 2),
        "Fit Time Sklearn (s)": round(sk_res["fit_time"], 2),
    })

summary_df = pd.DataFrame(summary_rows)
print("tabel komparasi svm dari nol vs scikit-learn:")
print(summary_df.to_string(index=False))

=== Linear (scikit-learn) ===
macro f1 : 0.8569
kelas 0  : prec=0.9039, rec=0.9098, f1=0.9069
kelas 1  : prec=0.8124, rec=0.8016, f1=0.8070
confusion matrix:
[[3349  332]
 [ 356 1438]]
waktu fit: 33.73s | waktu predict: 0.838s

=== Polynomial (scikit-learn) ===
macro f1 : 0.8550
kelas 0  : prec=0.9024, rec=0.9090, f1=0.9057
kelas 1  : prec=0.8104, rec=0.7982, f1=0.8043
confusion matrix:
[[3346  335]
 [ 362 1432]]
waktu fit: 22.16s | waktu predict: 1.750s

=== Gaussian RBF (scikit-learn) ===
macro f1 : 0.8575
kelas 0  : prec=0.8994, rec=0.9180, f1=0.9086
kelas 1  : prec=0.8242, rec=0.7893, f1=0.8064
confusion matrix:
[[3379  302]
 [ 378 1416]]
waktu fit: 37.59s | waktu predict: 5.460s

tabel komparasi svm dari nol vs scikit-learn:
      Kernel  Macro F1 (Scratch)  Macro F1 (Sklearn)  Fit Time Scratch (s)  Fit Time Sklearn (s)
      Linear              0.8574              0.8569                  1.84                 33.73
  Polynomial              0.8520              0.8550              

## 5. pelatihan ulang konfigurasi terbaik pada seluruh data latih

In [ ]:
print("melatih ulang model terbaik pada seluruh data latih (27.373 baris)...")

# prapemroses pada seluruh data latih
prep_final = SVMPreprocessor(seed=42)
prep_final.fit(train_df)

X_train_full = prep_final.transform(train_df)
y_train_full = train_df["is_canceled"].to_numpy(dtype=np.int64)
X_test_full = prep_final.transform(test_df)

print(f"X_train_full: {X_train_full.shape}, X_test_full: {X_test_full.shape}")

# pemenang: gaussian rbf kernel (macro f1 = 0.8573)
model_final = SVMFromScratch(C=1.0, kernel="rbf", gamma="scale", threshold=-0.16, max_iter=45, n_components=1400, random_state=42)
t_start = time.perf_counter()
model_final.fit(X_train_full, y_train_full)
print(f"retrain selesai dalam {time.perf_counter() - t_start:.2f}s | support vectors: {model_final.n_support_}")

test_preds_initial = model_final.predict(X_test_full)
print(f"distribusi prediksi test: {np.bincount(test_preds_initial)}")

melatih ulang model terbaik pada seluruh data latih (27.373 baris)...
X_train_full: (27373, 90), X_test_full: (6844, 90)
retrain selesai dalam 17.39s | support vectors: 9532
distribusi prediksi test: [4630 2214]


## 6. penyimpanan artefak dan verifikasi muat ulang (save & load)

In [ ]:
model_save_dir = REPO_ROOT / "output" / "final" / "model" / "svm"
model_save_dir.mkdir(parents=True, exist_ok=True)
bundle_path = model_save_dir / "svm_pipeline.pkl"

# simpan artefak pipeline (preprocessor + model terlatih)
with open(bundle_path, "wb") as f:
    pickle.dump({"preprocessor": prep_final, "model": model_final}, f)

print(f"artefak berhasil disimpan di: {bundle_path} ({bundle_path.stat().st_size / (1024**2):.2f} MB)")

# muat kembali artefak pada sesi bersih
with open(bundle_path, "rb") as f:
    loaded_bundle = pickle.load(f)

loaded_prep = loaded_bundle["preprocessor"]
loaded_model = loaded_bundle["model"]

# uji konsistensi prediksi sebelum dan sesudah load
X_test_loaded = loaded_prep.transform(test_df)
test_preds_loaded = loaded_model.predict(X_test_loaded)

assert np.array_equal(test_preds_initial, test_preds_loaded), "prediksi berbeda setelah load!"
assert set(np.unique(test_preds_loaded)).issubset({0, 1}), "label bukan biner 0/1!"
assert len(test_preds_loaded) == len(test_df), "panjang prediksi tidak sama dengan test.csv!"

print("verifikasi sukses: prediksi 100% identik sebelum dan setelah artefak dimuat ulang.")

artefak berhasil disimpan di: D:\IF_Semester 5\AI\Tubes 2\IF3170_G03_FiveAgents\output\final\model\svm\svm_pipeline.pkl (1.92 MB)
verifikasi sukses: prediksi 100% identik sebelum dan setelah artefak dimuat ulang.


## 7. pembuatan berkas submisi kaggle (output/submission.csv)

In [ ]:
submission_df = pd.DataFrame({
    "id": test_df["id"].astype(str),
    "is_canceled": test_preds_loaded.astype(int)
})

# simpan di output/final/model/svm/ dan output/submission.csv
sub_svm_path = model_save_dir / "submission_svm.csv"
sub_global_path = REPO_ROOT / "output" / "submission.csv"

submission_df.to_csv(sub_svm_path, index=False)
submission_df.to_csv(sub_global_path, index=False)

print(f"submission tersimpan di: {sub_svm_path} dan {sub_global_path}")
print("pratinjau submission:")
print(submission_df.head(10).to_string(index=False))
print(f"jumlah baris: {len(submission_df)}, nilai null: {submission_df.isnull().sum().to_dict()}")
print(f"distribusi kelas submission: {submission_df['is_canceled'].value_counts().to_dict()}")

submission tersimpan di: D:\IF_Semester 5\AI\Tubes 2\IF3170_G03_FiveAgents\output\final\model\svm\submission_svm.csv dan D:\IF_Semester 5\AI\Tubes 2\IF3170_G03_FiveAgents\output\submission.csv
pratinjau submission:
      id  is_canceled
5cb6f473            1
c06b4ab3            1
22bf3fc3            0
25408a48            0
621586b6            0
82d7a4f9            1
69ba1757            0
b6178e55            0
80fec53f            0
d05d0bc3            1
jumlah baris: 6844, nilai null: {'id': 0, 'is_canceled': 0}
distribusi kelas submission: {0: 4630, 1: 2214}
